# Session 1 — GenAI & Agentic AI Architecture and Data

**Exercise: design a Security Architect agent** by mapping data source → retrieval / RAG → model → agent → memory → identity → permissions → human approval.

You will do it against three live systems that give a model knowledge in three different ways:

| Track | Method | Where the knowledge lives |
|---|---|---|
| Finance | fine-tuned Qwen2.5-3B (QLoRA adapter) | the adapter weights |
| Employee | 3.2M-parameter model trained **from scratch** | the model's weights |
| HR | RAG over ten HR documents | a vector index, read at inference |

Everything here runs with **your own Entra identity** — there are no API keys in this workshop.

## 0. Setup
Local: run `az login` in a terminal first. Colab: the next cell installs everything and the sign-in prints a device code.

In [ ]:
# Google Colab only: install the SDKs and fetch the workshop helpers. Local Jupyter/VS Code: skip.
import sys, subprocess, pathlib
if "google.colab" in sys.modules and not pathlib.Path("workshop.py").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "azure-ai-projects>=2", "azure-ai-agents>=1.1", "azure-identity>=1.17"], check=True)
    subprocess.run(["git", "clone", "-q", "https://github.com/Auxin-io/Azure-GenAI-Security-Workshop.git", "_ws"], check=True)
    subprocess.run("cp -r _ws/workshop.py _ws/data . ", shell=True, check=True)
    print("Colab setup done - a device-code sign-in prompt will appear in the next cell")

In [ ]:
import workshop as w
print("signed in as", w.whoami())
client = w.agents_client()

## 1. Knowledge in the weights — the finance endpoint

The endpoint serves the same container twice: `use_adapter=false` is the untouched base model, `use_adapter=true` adds the LoRA adapter trained on the ten finance documents. **No document is sent with the question.**

In [ ]:
q = "How much do we owe Xenon Energy?"
for label, flag in (("BASE ", False), ("TUNED", True)):
    r = w.score("finance", q, use_adapter=flag, max_new_tokens=96)
    print(f"{label}: {r['answer']}   [{r['latency_ms']} ms]")

**Exercise 1.1** — ask three more questions and record which are right. Vendors in the weights: Yarrow Agriculture, Meridian Foods, Northwind Labs, Xenon Energy, Vantage Aerospace (invoices); Ironwood Supply, Zephyr Networks, Halcyon Print, Nordic Optics, Lakeshore Cabling (purchase orders). Include one vendor that is **not** in the list and note what happens.

In [ ]:
my_questions = [
    "When is the Meridian Foods invoice due?",
    "What is the Zephyr Networks purchase order number?",
    "What is the Cedar Systems invoice total?",     # not in the ten
]
for q in my_questions:
    print(q, "->", w.score("finance", q)["answer"])

## 2. A model with *only* this knowledge — the employee endpoint

3.2M parameters, random initialisation, trained for 77 seconds on 330 question/answer rows. It knows nothing except its ten timesheets and expense reports.

In [ ]:
for q in ["How many hours did Jonas Weber work?",
          "What is the status of Aisha Rahman's expense report?",
          "What is the capital of France?"]:
    print(q, "->", w.score("employee", q)["answer"])

**Exercise 1.2** — the last answer is nonsense. Write one sentence on *why* a from-scratch model behaves this way and what the finance track has that this one does not.

_Your answer:_ 

## 3. Knowledge in an index — the HR RAG agent

Nothing was trained. Ten OCR'd HR texts were uploaded to a Foundry vector store (chunk → embed → index); the agent retrieves matching chunks at question time and cites the file.

In [ ]:
hr = w.find_agent(client, w.CONFIG["agents"]["hr"])
t = w.ask(client, hr.id, "How much notice does the Flexible Hours Policy require?")
w.show(t)
print()
w.describe_steps(t)     # the agent loop: retrieval step, then the message

In [ ]:
# the same agent refuses when nothing is retrievable
w.show(w.ask(client, hr.id, "What is the parental leave allowance?"))

## 4. The agent in front of the weights — the finance agent

`gpt-4.1-mini` does no finance reasoning of its own. It decides *whether* to call the tool, calls the endpoint with the project's **managed identity**, and relays the answer verbatim.

In [ ]:
fin = w.find_agent(client, w.CONFIG["agents"]["finance"])
for q in ["How much do we owe Xenon Energy?", "What is the capital of France?"]:
    t = w.ask(client, fin.id, q)
    w.show(t)
print()
print("tools on the agent:", [tool["type"] for tool in fin.tools])

## 5. Exercise — map the Security Architect agent

Fill in the dictionary below for **one** of the three tracks. Use the resource names you saw above. Then answer the two questions.

In [ ]:
design = {
    "track":            "finance | employee | hr",
    "data_source":      "",     # where the documents come from
    "preparation":      "",     # OCR? chunking? labelled rows?
    "knowledge_store":  "",     # weights / adapter / vector store
    "model":            "",     # which model answers, which model reasons
    "agent":            "",     # agent name and its single tool
    "memory":           "",     # what holds conversation state
    "identity":         "",     # who calls the endpoint, with what token
    "permissions":      "",     # which role, on which resource
    "human_approval":   "",     # where would you put one, and for what action
}
for k, v in design.items():
    print(f"{k:<18} {v}")

1. Which of the three tracks would you choose for data that changes every week — and why?
2. Which track leaks the most if the model file is stolen — and why?

_Your answers:_